# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsahmed39/FlyRank--Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm going freestyle, though it sits close to Lane 2. The reason I didn't just take Lane 2 because it is the starter's own label, `trend_direction`, is computed over the full 90 day window, so it can call a page "stable" or even "up" even when that page just dropped hard in the last 30 days. For an editor deciding what to look at *this week*, "generally fine over the quarter" isn't the same question as "still losing ground right now." So instead of reusing `trend_direction`, I want to define my own forward looking signal compare `prev_30d` traffic to `last_30d` traffic directly, and rank on that recent movement instead of the pre-baked bucket. Section 3 shows this actually disagrees with `trend_direction` often enough to matter.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision it improves:** how a content editor spends limited review hours instead of scanning items in no particular order, or trusting a 90 day bucket that can be stale, they get a ranked shortlist.

**Action someone takes:** the editor opens the top ranked items and updates, rewrites, or otherwise refreshes them. Output is a ranked list with reasons, not a single score.

**Cost of a wrong call:**
- False positive (ranked high, wasn't actually still declining): editor time wasted on a page that didn't need it, while a page that's actually sliding waits.
- False negative (missed, but quietly still declining): that page keeps losing clicks/position until someone notices by accident, and since traffic loss is concentrated in a small share of items (Section 3), missing the wrong one is expensive.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Loading the starter CSV and checking whether a recent 30 day drop signal actually differs from the existing `trend_direction` bucket and whether the traffic at stake is concentrated enough to make prioritization worth doing.

In [2]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

# Only look at items that had clicks to lose in the prior window
sub = df.dropna(subset=["clicks_prev_30d", "clicks_last_30d"]).copy()
sub = sub[sub["clicks_prev_30d"] > 0]
sub["pct_change_30d"] = (sub["clicks_last_30d"] - sub["clicks_prev_30d"]) / sub["clicks_prev_30d"]
sub["recent_drop"] = sub["pct_change_30d"] <= -0.20

print(f"Eligible items (had clicks in prev_30d): {len(sub):,}")
print(f"Items with a >20% clicks drop in the most recent 30 days: {sub['recent_drop'].sum():,} "
      f"({100*sub['recent_drop'].mean():.1f}%)")

# Does this disagree with the existing trend_direction bucket?
mismatch = pd.crosstab(sub["recent_drop"], sub["trend_direction"])
print("\nrecent_drop (my label) vs trend_direction (90-day bucket):")
print(mismatch)

missed = mismatch.loc[True, ["stable", "up"]].sum()
print(f"\nItems the 90-day bucket calls 'stable' or 'up', but that actually dropped >20% "
      f"in the last 30 days: {missed:,}")

# How concentrated is traffic at risk among the recently-declining group?
declining = sub[sub["recent_drop"]].sort_values("clicks_prev_30d", ascending=False)
top10 = declining.head(int(len(declining) * 0.10))
share = 100 * top10["clicks_prev_30d"].sum() / declining["clicks_prev_30d"].sum()
print(f"\nTop 10% of recently-declining items by prior traffic hold "
      f"{share:.1f}% of the clicks at risk in that group.")

Eligible items (had clicks in prev_30d): 11,759
Items with a >20% clicks drop in the most recent 30 days: 6,147 (52.3%)

recent_drop (my label) vs trend_direction (90-day bucket):
trend_direction  down  stable    up
recent_drop                        
False            2590    1945  1077
True             4251    1405   491

Items the 90-day bucket calls 'stable' or 'up', but that actually dropped >20% in the last 30 days: 1,896

Top 10% of recently-declining items by prior traffic hold 65.2% of the clicks at risk in that group.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** which items showed a real, observed drop in clicks/impressions between two recent 30 day windows, and a ranked shortlist built from that plus exposure and position context.

**What I can't claim:**
- That refreshing a page will cause it to recover that needs an actual experiment, which I don't have here.
- That I've predicted or reverse engineered anything about Google's ranking algorithm I'm only working with FlyRank's own observed impressions/position data.
- That "recently declining" by my threshold is some absolute truth it's a definition I chose, and a different threshold would move some items in and out of the list. I'll say so explicitly and show what changes if the threshold moves.
- That my ranking is better than the existing rule that's exactly what Weeks 5-9 (baseline, model, validation) are for. Right now this is a framing exercise, not a result.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.a


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.